<a href="https://www.kaggle.com/code/saimanudar/cse438-project-updated?scriptVersionId=309342499" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
!pip install torch==2.2.0+cu118 torchvision==0.17.0+cu118 \
    --index-url https://download.pytorch.org/whl/cu118 -q
!pip install numpy==1.26.4 --force-reinstall -q
!pip install pillow -q

# Verify
import numpy as np
import torch
from PIL import Image
print(f"numpy  : {np.__version__}")
print(f"torch  : {torch.__version__}")
print(f"CUDA   : {torch.version.cuda}")

import io
img = Image.new('RGB', (224, 224), color=(128, 64, 32))
arr = np.array(img)
t   = torch.from_numpy(arr)
print(f" PIL→numpy→torch OK: {t.shape}")

# GPU test
x = torch.randn(3, 3).cuda()
y = x @ x
print(f" GPU OK: {y.shape}")
print("\n All fixed! Now: Run → Restart & Clear Output → Run All")

In [ ]:
!pip install torch==2.2.0+cu118 torchvision==0.17.0+cu118 \
    --index-url https://download.pytorch.org/whl/cu118 -q
print(" Done!")

In [ ]:
!pip install timm xgboost scikit-learn torchmetrics seaborn matplotlib tqdm -q
print('Packages installed.')

## Cell 2: Imports & Configuration

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

x = torch.randn(3, 3).cuda()
y = x @ x
print(" GPU Test Passed!", y.shape)

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import timm
from collections import Counter
from PIL import Image
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────
DATASET_ROOT = '/kaggle/input/datasets/saimanudar/cse438/Resized Image'
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_CLASSES  = 4
EPOCHS       = 30
LR           = 1e-4
SEED         = 42
DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLASS_NAMES  = ['Camphor', 'HariTaki', 'Neem', 'Sojina']

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device : {DEVICE}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU Capability: {torch.cuda.get_device_capability(0)}')

## Cell 3: EDA — Class Distribution

In [ ]:
CLASS_INFO = {
    'Camphor'  : 2396,
    'HariTaki' : 2421,
    'Neem'     : 3563,
    'Sojina'   : 2478,
}
classes = list(CLASS_INFO.keys())
counts  = list(CLASS_INFO.values())

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(classes)))
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(classes, counts, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Number of Images')
ax.set_title('AI-MedLeafX — Class Distribution (4 Classes)', fontsize=14, fontweight='bold')
ax.axhline(np.mean(counts), color='red', linestyle='--', label=f'Mean = {np.mean(counts):.0f}')
ax.legend()
for bar, count in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            str(count), ha='center', va='bottom', fontsize=10)
plt.tight_layout(); plt.show()
print(f'Total          : {sum(counts)}')
print(f'Imbalance ratio: {max(counts)/min(counts):.2f}x  ← mild imbalance')

## Cell 4: Augmentation & Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.25, contrast=0.25,
                            saturation=0.25, hue=0.08),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.12)),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
print('Transforms defined.')

## Cell 5: Load Dataset & Auto Split (80/10/10)

In [ ]:
# TransformSubset: allows different transforms for train vs val/test
class TransformSubset(Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img_path, label = self.dataset.samples[self.indices[idx]]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# Load full dataset (no transform yet)
full_dataset = datasets.ImageFolder(DATASET_ROOT, transform=val_test_transforms)
all_targets  = [label for _, label in full_dataset.samples]
class_counts = Counter(all_targets)

print(f'Total images : {len(full_dataset)}')
print(f'Classes      : {full_dataset.classes}')
print(f'Class counts : {dict(class_counts)}')

# Stratified 80/10/10 split
indices = list(range(len(full_dataset)))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.20, stratify=all_targets, random_state=SEED)
temp_targets = [all_targets[i] for i in temp_idx]
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_targets, random_state=SEED)

print(f'\nSplit → Train:{len(train_idx)} | Val:{len(val_idx)} | Test:{len(test_idx)}')

# Create subsets with proper transforms
train_dataset = TransformSubset(full_dataset, train_idx, train_transforms)
val_dataset   = TransformSubset(full_dataset, val_idx,   val_test_transforms)
test_dataset  = TransformSubset(full_dataset, test_idx,  val_test_transforms)

# WeightedRandomSampler for train loader
train_targets_list = [all_targets[i] for i in train_idx]
sample_weights     = [1.0 / class_counts[t] for t in train_targets_list]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, pin_memory=True)

# Class weights for loss function
actual_weights = compute_class_weight('balanced',
                                       classes=np.unique(train_targets_list),
                                       y=np.array(train_targets_list))
CLASS_WEIGHTS = torch.tensor(actual_weights, dtype=torch.float)  
print(f'\nClass weights (CPU): {CLASS_WEIGHTS}') 

# Sanity check
imgs, labels = next(iter(train_loader))
print(f'Batch shape  : {imgs.shape}')
print(' DataLoaders ready!')

## Cell 6: Class Distribution (After Weighted Sampler)

In [ ]:
# Count class distribution in 1 epoch of training (sampled)
sampled_counts = Counter()
for _, lbls in train_loader:
    for l in lbls.numpy():
        sampled_counts[int(l)] += 1

orig_counts    = [class_counts[i] for i in range(NUM_CLASSES)]
sampled_list   = [sampled_counts[i] for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, color in zip(
    axes,
    [orig_counts, sampled_list],
    ['Original Distribution', 'After WeightedSampler (1 epoch)'],
    ['#3498db', '#2ecc71']
):
    bars = ax.bar(CLASS_NAMES, data, color=color, edgecolor='black')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Count')
    for bar, c in zip(bars, data):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+10, str(c), ha='center', fontsize=9)
plt.suptitle('Imbalance Handling — AI-MedLeafX', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Original imbalance  : {max(orig_counts)/min(orig_counts):.2f}x')
print(f'Sampler imbalance   : {max(sampled_list)/max(min(sampled_list),1):.2f}x  ← much better!')

## Cell 7: Focal Loss & Early Stopping

In [ ]:
class FocalLoss(nn.Module):
    """Focuses training on hard/minority samples. Better than CE for imbalanced data."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce    = nn.CrossEntropyLoss(weight=alpha, reduction='none')

    def forward(self, inputs, targets):
        ce   = self.ce(inputs, targets)
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.001, mode='max'):
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.best      = -np.inf if mode == 'max' else np.inf
        self.counter   = 0
        self.triggered = False

    def step(self, metric):
        improved = (
            (self.mode == 'max' and metric > self.best + self.min_delta) or
            (self.mode == 'min' and metric < self.best - self.min_delta)
        )
        if improved:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return improved

print(' FocalLoss & EarlyStopping defined.')

## Cell 8: Training Engine

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

def train_one_epoch(model, loader, optimizer, criterion, scaler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if scaler:
            with torch.cuda.amp.autocast():
                out  = model(imgs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = out.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss/total, correct/total, f1_score(all_labels, all_preds, average='weighted')


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out  = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * imgs.size(0)
        preds = out.argmax(1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss/total, correct/total, f1_score(all_labels, all_preds, average='weighted')


def train_model(model, train_loader, val_loader, exp_name,
                lr=1e-4, epochs=EPOCHS, unfreeze_ep=5):
    model = model.to(DEVICE)
    criterion = FocalLoss(alpha=CLASS_WEIGHTS.to(DEVICE), gamma=2.0)
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4)
    scheduler = OneCycleLR(
        optimizer, max_lr=lr*10,
        epochs=epochs, steps_per_epoch=len(train_loader),
        pct_start=0.3, anneal_strategy='cos')

    # AMP disabled for compatibility (avoids CUDA kernel version errors)
    scaler = None

    stopper = EarlyStopping(patience=8, mode='max')
    best_w  = None
    history = {k: [] for k in ['tl','vl','ta','va','tf','vf']}

    print('=' * 65)
    print(f'  {exp_name}  |  Epochs:{epochs}  |  LR:{lr}  |  {DEVICE}')
    print('=' * 65)

    for ep in range(1, epochs+1):
        t0 = time.time()

        # Unfreeze all at unfreeze_ep
        if ep == unfreeze_ep:
            print(f'  [Ep {ep}] Unfreezing all layers...')
            for p in model.parameters():
                p.requires_grad = True
            optimizer = AdamW(model.parameters(), lr=lr/10, weight_decay=1e-4)

        tl, ta, tf = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        scheduler.step()
        vl, va, vf = validate(model, val_loader, criterion)

        for key, val in zip(['tl','vl','ta','va','tf','vf'], [tl,vl,ta,va,tf,vf]):
            history[key].append(val)

        print(f'Ep{ep:3d}/{epochs} | '
              f'T-Loss:{tl:.4f} T-Acc:{ta:.3f} T-F1:{tf:.3f} | '
              f'V-Loss:{vl:.4f} V-Acc:{va:.3f} V-F1:{vf:.3f} | '
              f'{time.time()-t0:.1f}s')

        if stopper.step(vf):
            best_w = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_w, f'best_{exp_name}.pth')
            print(f'  ✓ Saved best → best_{exp_name}.pth')
        if stopper.triggered:
            print(f'  Early stop at epoch {ep}.'); break

    model.load_state_dict(best_w)
    return model, history


def plot_history(history, name):
    eps = range(1, len(history['tl'])+1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (tk, vk), title in zip(
        axes,
        [('tl','vl'),('ta','va'),('tf','vf')],
        ['Loss', 'Accuracy', 'F1 Score']
    ):
        ax.plot(eps, history[tk], 'b-o', markersize=3, label='Train')
        ax.plot(eps, history[vk], 'r-o', markersize=3, label='Val')
        ax.set_title(f'{name} — {title}', fontweight='bold')
        ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'history_{name}.png', dpi=150)
    plt.show()

print(' Training engine ready.')
print(' AMP disabled for CUDA compatibility.')

## Cell 9: Model A — EfficientNet-B3 + Transformer

In [ ]:
# ── Debug cell: actual feature dim check ──────────────────
import timm, torch

cnn_test = timm.create_model('efficientnet_b3', pretrained=False,
                               features_only=True, out_indices=[4])
x_test   = torch.randn(1, 3, 224, 224)
feats    = cnn_test(x_test)
print(f'Feature shape: {feats[0].shape}')



In [ ]:
class CNNTransformerHybrid(nn.Module):
    """EfficientNet-B3 (local features) + Transformer (global context)."""
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.4):
        super().__init__()
        self.cnn = timm.create_model('efficientnet_b3', pretrained=True,
                                      features_only=True, out_indices=[4])

        cnn_dim  = 384   # ← actual dim (confirmed from debug)
        seq_len  = 49    # 7×7 = 49

        self.proj = nn.Sequential(
            nn.AdaptiveAvgPool2d((7, 7)),
            nn.Flatten(2)              # [B, 384, 49]
        )
        self.pos_embed = nn.Parameter(
            torch.randn(1, seq_len, cnn_dim) * 0.02)  # [1, 49, 384]

        # nhead must divide cnn_dim evenly: 384 / 8 = 48 ✓
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cnn_dim, nhead=8,
            dim_feedforward=1024,
            dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)

        self.classifier = nn.Sequential(
            nn.LayerNorm(cnn_dim), nn.Dropout(dropout),
            nn.Linear(cnn_dim, 256), nn.GELU(),
            nn.Dropout(dropout/2),
            nn.Linear(256, num_classes))

    def forward(self, x):
        f   = self.cnn(x)[0]                     # [B, 384, 7, 7]
        seq = self.proj(f).permute(0, 2, 1)      # [B, 49, 384]
        seq = seq + self.pos_embed
        out = self.transformer(seq).mean(dim=1)  # [B, 384]
        return self.classifier(out)

# ── Test ──────────────────────────────────────────────────
_m = CNNTransformerHybrid(num_classes=NUM_CLASSES)
_x = torch.randn(2, 3, 224, 224)
print(f'Output shape : {_m(_x).shape}')   # → [2, 4]
print(f'Total params : {sum(p.numel() for p in _m.parameters()):,}')
del _m, _x

## Cell 10: Train Experiment A

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("GPU Capability:", torch.cuda.get_device_capability(0))

In [ ]:
model_a = CNNTransformerHybrid(num_classes=NUM_CLASSES).to(DEVICE)

# Freeze CNN backbone for warm-up (first unfreeze_ep epochs)
for p in model_a.cnn.parameters():
    p.requires_grad = False
print('CNN backbone frozen for warm-up.')

model_a, hist_a = train_model(
    model_a, train_loader, val_loader,
    exp_name='ExpA_CNN_Transformer',
    lr=LR, epochs=EPOCHS, unfreeze_ep=5
)
plot_history(hist_a, 'ExpA')

## Cell 11: Model B — EfficientNet-B3 → XGBoost

In [ ]:
import xgboost as xgb

class CNNFeatureExtractor(nn.Module):
    """EfficientNet-B3 without head — outputs 1536-d embeddings."""
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b3', pretrained=True,
            num_classes=0, global_pool='avg')

    def forward(self, x):
        return self.backbone(x)  # [B, 1536]


@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    model.to(DEVICE)
    feats, lbls = [], []
    for imgs, labels in loader:
        feats.append(model(imgs.to(DEVICE)).cpu().numpy())
        lbls.append(labels.numpy())
    return np.concatenate(feats), np.concatenate(lbls)


feat_model = CNNFeatureExtractor()
print('Extracting train features...')
train_feats, train_labels_xgb = extract_features(feat_model, train_loader)
print('Extracting val features...')
val_feats,   val_labels_xgb   = extract_features(feat_model, val_loader)
print('Extracting test features...')
test_feats,  test_labels_xgb  = extract_features(feat_model, test_loader)
print(f'Feature shape: {train_feats.shape}  ← [N, 1536 CNN embedding]')

## Cell 12: Train XGBoost Head

In [ ]:
clf_b = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', objective='multi:softprob',
    num_class=NUM_CLASSES,
    tree_method='hist',   # 'gpu_hist' if supported, else 'hist'
    device='cuda' if torch.cuda.is_available() else 'cpu',
    random_state=SEED, n_jobs=-1, verbosity=1
)
clf_b.fit(
    train_feats, train_labels_xgb,
    eval_set=[(val_feats, val_labels_xgb)],
    verbose=50
)
val_preds_b = clf_b.predict(val_feats)
print(f'\nXGBoost Val Accuracy : {accuracy_score(val_labels_xgb, val_preds_b):.4f}')
print(f'XGBoost Val F1 (w)   : {f1_score(val_labels_xgb, val_preds_b, average="weighted"):.4f}')

## Cell 13: Model C — MobileNetV3 + SE Attention (Lightweight)

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention."""
    def __init__(self, ch, r=16):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(ch, ch//r, bias=False), nn.ReLU(),
            nn.Linear(ch//r, ch, bias=False), nn.Sigmoid())

    def forward(self, x):
        return x * self.se(x).view(x.size(0), x.size(1), 1, 1)


class LightweightSEHybrid(nn.Module):
    """MobileNetV3-Small + SE blocks. ~4M params."""
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'mobilenetv3_small_100', pretrained=True,
            features_only=True, out_indices=[3, 4])
        self.se3  = SEBlock(48,  r=8)
        self.se4  = SEBlock(576, r=16)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fusion = nn.Sequential(
            nn.Flatten(),
            nn.Linear(48+576, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(256, num_classes))

    def forward(self, x):
        f  = self.backbone(x)
        f3 = self.pool(self.se3(f[0])).flatten(1)
        f4 = self.pool(self.se4(f[1])).flatten(1)
        return self.fusion(torch.cat([f3, f4], dim=1))


model_c = LightweightSEHybrid(num_classes=NUM_CLASSES).to(DEVICE)
print(f'Exp C — params: {sum(p.numel() for p in model_c.parameters()):,}  ← Lightweight!')
print(f'Output shape  : {model_c(torch.randn(2,3,224,224).to(DEVICE)).shape}')

## Cell 14: Train Experiment C

In [ ]:
model_c, hist_c = train_model(
    model_c, train_loader, val_loader,
    exp_name='ExpC_Lightweight_SE',
    lr=LR, epochs=EPOCHS, unfreeze_ep=5
)
plot_history(hist_c, 'ExpC')

## Cell 15: Model D — Ensemble (EfficientNet + ResNet50 + DenseNet121)

In [ ]:
class EnsembleHybrid(nn.Module):
    """3 CNN backbones fused with attention gating."""
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.4):
        super().__init__()
        self.eff = timm.create_model('efficientnet_b0', pretrained=True,
                                      num_classes=0, global_pool='avg')  # 1280
        self.res = timm.create_model('resnet50',       pretrained=True,
                                      num_classes=0, global_pool='avg')  # 2048
        self.den = timm.create_model('densenet121',    pretrained=True,
                                      num_classes=0, global_pool='avg')  # 1024
        pd = 512
        self.pe   = nn.Sequential(nn.Linear(1280, pd), nn.ReLU())
        self.pr   = nn.Sequential(nn.Linear(2048, pd), nn.ReLU())
        self.pden = nn.Sequential(nn.Linear(1024, pd), nn.ReLU())
        self.gate = nn.Sequential(nn.Linear(pd*3, 3), nn.Softmax(dim=1))
        self.head = nn.Sequential(
            nn.Linear(pd, 256), nn.BatchNorm1d(256), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(256, num_classes))

    def forward(self, x):
        e = self.pe(self.eff(x))
        r = self.pr(self.res(x))
        d = self.pden(self.den(x))
        w = self.gate(torch.cat([e, r, d], dim=1))
        fused = w[:,0:1]*e + w[:,1:2]*r + w[:,2:3]*d
        return self.head(fused)


model_d = EnsembleHybrid(num_classes=NUM_CLASSES).to(DEVICE)
print(f'Exp D — params: {sum(p.numel() for p in model_d.parameters()):,}')
print(f'Output shape  : {model_d(torch.randn(2,3,224,224).to(DEVICE)).shape}')

## Cell 16: Train Experiment D

In [ ]:
model_d, hist_d = train_model(
    model_d, train_loader, val_loader,
    exp_name='ExpD_Ensemble',
    lr=LR/2, epochs=EPOCHS, unfreeze_ep=8
)
plot_history(hist_d, 'ExpD')

## Cell 17: Evaluate All Models on Test Set

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, name):
    model.eval(); model.to(DEVICE)
    preds, labels = [], []
    for imgs, lbls in loader:
        preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    preds  = np.array(preds)
    labels = np.array(labels)
    acc = accuracy_score(labels, preds)
    f1w = f1_score(labels, preds, average='weighted')
    f1m = f1_score(labels, preds, average='macro')
    print(f'\n===== {name} =====')
    print(f'Accuracy     : {acc*100:.2f}%')
    print(f'F1 Weighted  : {f1w:.4f}')
    print(f'F1 Macro     : {f1m:.4f}')
    print(classification_report(labels, preds, target_names=CLASS_NAMES))
    return {'model':name, 'accuracy':acc, 'f1_weighted':f1w, 'f1_macro':f1m,
            'y_true':labels, 'y_pred':preds}


def eval_xgboost(clf, feats, labels, name):
    preds = clf.predict(feats)
    acc = accuracy_score(labels, preds)
    f1w = f1_score(labels, preds, average='weighted')
    f1m = f1_score(labels, preds, average='macro')
    print(f'\n===== {name} =====')
    print(f'Accuracy    : {acc*100:.2f}%')
    print(f'F1 Weighted : {f1w:.4f}')
    print(f'F1 Macro    : {f1m:.4f}')
    print(classification_report(labels, preds, target_names=CLASS_NAMES))
    return {'model':name, 'accuracy':acc, 'f1_weighted':f1w, 'f1_macro':f1m,
            'y_true':labels, 'y_pred':preds}


print('Evaluating all models on TEST set...')
res_a = evaluate_model(model_a, test_loader, 'ExpA: CNN+Transformer')
res_b = eval_xgboost(clf_b, test_feats, test_labels_xgb, 'ExpB: CNN+XGBoost')
res_c = evaluate_model(model_c, test_loader, 'ExpC: Lightweight SE')
res_d = evaluate_model(model_d, test_loader, 'ExpD: Ensemble')

## Cell 18: Confusion Matrices

In [ ]:
def plot_cm(y_true, y_pred, title):
    cm   = confusion_matrix(y_true, y_pred)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, data, fmt, t in zip(
        axes, [cm, cm_n], ['d', '.2f'], ['Count', 'Normalized']
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    linewidths=0.5, ax=ax)
        ax.set_title(f'{title} — {t}', fontsize=12)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(f'cm_{title.replace(" ","_")}.png', dpi=120)
    plt.show()

for r in [res_a, res_b, res_c, res_d]:
    plot_cm(r['y_true'], r['y_pred'], r['model'])

## Cell 19: Per-Class F1 Bar Charts

In [ ]:
def plot_f1_per_class(y_true, y_pred, title):
    f1s    = f1_score(y_true, y_pred, average=None)
    colors = ['#2ecc71' if f>=0.90 else '#f39c12' if f>=0.75 else '#e74c3c'
               for f in f1s]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(CLASS_NAMES, f1s, color=colors, edgecolor='black', lw=0.5)
    ax.set_ylim(0, 1.1)
    ax.axhline(0.90, ls='--', color='green',  alpha=0.6, label='0.90 threshold')
    ax.axhline(0.75, ls='--', color='orange', alpha=0.6, label='0.75 threshold')
    ax.set_title(f'{title} — Per-Class F1', fontsize=13, fontweight='bold')
    ax.set_ylabel('F1 Score'); ax.legend()
    for bar, f in zip(bars, f1s):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+0.01, f'{f:.2f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'f1_{title.replace(" ","_")}.png', dpi=120)
    plt.show()

for r in [res_a, res_b, res_c, res_d]:
    plot_f1_per_class(r['y_true'], r['y_pred'], r['model'])

## Cell 20: Final Model Comparison

In [ ]:
results = [res_a, res_b, res_c, res_d]

df = pd.DataFrame([{
    'Model'       : r['model'],
    'Accuracy'    : f"{r['accuracy']*100:.2f}%",
    'F1 Weighted' : f"{r['f1_weighted']:.4f}",
    'F1 Macro'    : f"{r['f1_macro']:.4f}",
} for r in results])

print('='*60)
print('  FINAL EXPERIMENT COMPARISON')
print('='*60)
print(df.to_string(index=False))
df.to_csv('experiment_comparison.csv', index=False)

# Bar chart comparison
models = [r['model'].split(':')[0] for r in results]
x = np.arange(len(models))
w = 0.25
fig, ax = plt.subplots(figsize=(12, 6))
for i, (key, label, color) in enumerate([
    ('accuracy',    'Accuracy',    '#3498db'),
    ('f1_weighted', 'F1 Weighted', '#2ecc71'),
    ('f1_macro',    'F1 Macro',    '#e67e22')
]):
    vals = [r[key] for r in results]
    bars = ax.bar(x + i*w, vals, w, label=label, color=color,
                   edgecolor='black', lw=0.5)
    for j, v in enumerate(vals):
        ax.text(x[j]+i*w, v+0.005, f'{v:.3f}',
                ha='center', fontsize=8, rotation=45)

ax.set_xticks(x + w)
ax.set_xticklabels(models, rotation=10)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Hybrid Model Comparison — AI-MedLeafX (4-Class)',
              fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()
print('\n Done! Files saved: experiment_comparison.csv, model_comparison.png')

## Cell 21: Inference — Predict a Single Image

In [ ]:
def predict_image(model, image_path, model_name='Model'):
    """Predict class for a single leaf image and show top-4 confidence."""
    FULL_NAMES = ['Camphor', 'HariTaki', 'Neem', 'Sojina']
    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
    ])
    img    = Image.open(image_path).convert('RGB')
    tensor = tf(img).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1).squeeze().cpu().numpy()
    top_idx  = probs.argsort()[::-1]
    pred     = FULL_NAMES[top_idx[0]]
    conf     = probs[top_idx[0]] * 100

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(img); axes[0].axis('off')
    axes[0].set_title(f'{model_name}\nPrediction: {pred}\nConf: {conf:.1f}%',
                       fontsize=11, fontweight='bold')
    names  = [FULL_NAMES[i] for i in top_idx[::-1]]
    values = [probs[i]*100  for i in top_idx[::-1]]
    colors = ['#2ecc71' if i==top_idx[0] else '#bdc3c7' for i in top_idx[::-1]]
    axes[1].barh(names, values, color=colors, edgecolor='black', lw=0.5)
    axes[1].set_xlabel('Confidence (%)')
    axes[1].set_title('Class Probabilities')
    for i, v in enumerate(values):
        axes[1].text(v+0.5, i, f'{v:.1f}%', va='center', fontsize=9)
    plt.tight_layout(); plt.show()
    return pred, conf

# Example usage:
# pred, conf = predict_image(model_a, '/path/to/leaf.jpg', 'ExpA')
print('predict_image() ready!')
print('Usage: predict_image(model_a, "path/to/image.jpg", "ExpA")')